# EDA & Preprocessing — CFPB Consumer Complaints

This notebook explores the **CFPB Consumer Complaint** dataset for the CrediTrust RAG chatbot.

**Goals**
1. Load the full CFPB dataset.
2. Examine the complaint distribution by product.
3. Count complaints **with** and **without** a consumer narrative.
4. Visualize the distribution of narrative lengths.
5. Identify very short and very long narratives.
6. Filter to the four target products and inspect the result.
7. Summarize key EDA findings.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the project's src package importable from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 5)

# Column names in the CFPB dataset
PRODUCT_COL = "Product"
NARRATIVE_COL = "Consumer complaint narrative"

# Path to the raw CFPB complaints CSV (update the filename as needed)
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "complaints.csv"
RAW_DATA_PATH

## 1. Load the full CFPB dataset

We load the complete dataset (no row sampling) so the EDA reflects the true distribution.
The CFPB file is large, so we read it once and inspect its shape, columns, and a few rows.

In [ ]:
# Load the full dataset. low_memory=False avoids dtype-guessing warnings on large files.
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("\nColumns:")
print(list(df.columns))
df.head()

## 2. Complaint distribution by product

How many complaints fall under each product category? This tells us which products
dominate the dataset and helps validate our target-product selection.

In [ ]:
product_counts = df[PRODUCT_COL].value_counts()
print("Complaints per product:\n")
print(product_counts)

ax = product_counts.plot(kind="barh", color="steelblue")
ax.invert_yaxis()
ax.set_title("Complaint Distribution by Product")
ax.set_xlabel("Number of complaints")
ax.set_ylabel("Product")
plt.tight_layout()
plt.show()

## 3. Complaints with vs. without a narrative

The RAG system can only use complaints that contain a free-text narrative. Here we count
how many records have a narrative versus how many are empty/missing.

In [ ]:
# A narrative is "present" if it is non-null and not just whitespace
has_narrative = df[NARRATIVE_COL].notna() & (df[NARRATIVE_COL].str.strip() != "")

n_with = int(has_narrative.sum())
n_without = int((~has_narrative).sum())
total = len(df)

print(f"With narrative:    {n_with:,} ({n_with / total:.1%})")
print(f"Without narrative: {n_without:,} ({n_without / total:.1%})")

ax = pd.Series({"With narrative": n_with, "Without narrative": n_without}).plot(
    kind="bar", color=["seagreen", "indianred"]
)
ax.set_title("Complaints With vs. Without a Narrative")
ax.set_ylabel("Number of complaints")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Narrative length distribution

We measure narrative length in **words** to understand typical complaint sizes. This
informs our chunking strategy (chunk size and overlap) downstream.

In [ ]:
# Work only with rows that actually have a narrative
narratives = df.loc[has_narrative, NARRATIVE_COL]
word_counts = narratives.str.split().str.len()

print("Narrative length (words) — summary statistics:")
print(word_counts.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(word_counts, bins=60, color="slateblue", edgecolor="white")
axes[0].set_title("Narrative Length Distribution (words)")
axes[0].set_xlabel("Words per narrative")
axes[0].set_ylabel("Frequency")

# Zoom into the bulk of the data (clip the long tail for readability)
clip = word_counts.quantile(0.99)
axes[1].hist(word_counts[word_counts <= clip], bins=60, color="mediumseagreen", edgecolor="white")
axes[1].set_title(f"Narrative Length (<= 99th pct = {clip:.0f} words)")
axes[1].set_xlabel("Words per narrative")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 5. Very short and very long narratives

Extremely short narratives carry little semantic signal, while extremely long ones may
need aggressive chunking. We flag both ends using word-count thresholds.

In [ ]:
SHORT_THRESHOLD = 5      # words — likely too little signal
LONG_THRESHOLD = 500     # words — very long, will require multiple chunks

short_mask = word_counts < SHORT_THRESHOLD
long_mask = word_counts > LONG_THRESHOLD

print(f"Very short narratives (< {SHORT_THRESHOLD} words): {short_mask.sum():,}")
print(f"Very long narratives  (> {LONG_THRESHOLD} words): {long_mask.sum():,}")

print("\nExamples of VERY SHORT narratives:")
for text in narratives[short_mask].head(5):
    print(f"  - {text!r}")

print("\nExamples of VERY LONG narratives (first 200 chars):")
for text in narratives[long_mask].head(3):
    print(f"  - {text[:200]}...")

## 6. Filter to the four target products

CrediTrust focuses on four core products. We map the relevant CFPB product labels to
these targets, keep only rows **with** a narrative, and inspect the resulting subset.

> Adjust `TARGET_PRODUCTS` if the exact CFPB labels differ in your dataset version.

In [ ]:
# Substrings used to match CFPB product labels to our four target categories.
# CFPB labels vary (e.g. "Credit card or prepaid card"), so we match on keywords.
TARGET_PRODUCT_PATTERNS = {
    "Credit card": "credit card",
    "Personal loan": "personal loan",
    "Savings account": "savings account",
    "Money transfer": "money transfer",
}

product_lower = df[PRODUCT_COL].fillna("").str.lower()
target_mask = pd.Series(False, index=df.index)
for label, pattern in TARGET_PRODUCT_PATTERNS.items():
    target_mask |= product_lower.str.contains(pattern, na=False)

# Final filtered dataset: target products AND has a narrative
df_filtered = df[target_mask & has_narrative].copy()

print(f"Rows before filtering:           {len(df):,}")
print(f"Rows in target products:         {int(target_mask.sum()):,}")
print(f"Rows after narrative filter too: {len(df_filtered):,}")
print("\nProduct breakdown after filtering:")
print(df_filtered[PRODUCT_COL].value_counts())

In [ ]:
# Visualize the filtered distribution and persist the cleaned subset for later steps
ax = df_filtered[PRODUCT_COL].value_counts().plot(kind="bar", color="darkorange")
ax.set_title("Filtered Complaints by Product (target products, with narrative)")
ax.set_ylabel("Number of complaints")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

processed_path = PROJECT_ROOT / "data" / "processed" / "filtered_complaints.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(processed_path, index=False)
print(f"Saved filtered dataset -> {processed_path}")

## 7. Key EDA findings

> _Fill in the bracketed numbers after running the notebook on your data version._

- **Dataset size:** the full CFPB export contains **[N]** complaints across **[K]** product categories.
- **Product distribution:** complaints are highly imbalanced across products; a few categories dominate the volume.
- **Narrative coverage:** only **[~X%]** of complaints include a free-text narrative — the rest are unusable for RAG and are dropped.
- **Narrative length:** lengths are **right-skewed** — most narratives are short-to-moderate (median ≈ **[M]** words) with a long tail of very detailed complaints.
- **Outliers:** **[a]** narratives are extremely short (< 5 words, low signal) and **[b]** are very long (> 500 words, will span multiple chunks).
- **Target subset:** filtering to the four target products *with* a narrative yields **[F]** records — the working dataset for chunking, embedding, and retrieval.

**Implications for the pipeline**
- Drop complaints without narratives before indexing.
- Choose a chunk size/overlap that accommodates the median length while splitting the long tail.
- Be mindful of product imbalance when interpreting retrieval results.